# 4 cm 큐브: 한 시점의 표면 → GraspNet → grasp / pre-grasp

기본 입력은 정면이며 카메라 위치를 바꿔 비스듬한 단일 시점도 만들 수 있습니다. 보이는 표면의 N×3 점을 camera frame으로 변환해 공식 baseline에 넣습니다. 숨은 면은 입력하지 않습니다. 실제 2D 픽셀로부터 전체 큐브를 복원하는 기능은 없습니다. 이 합성 실험은 가상 카메라 위치와 알려진 4 cm 형상으로 깊이를 정의합니다.

GPU runtime을 선택하고 셀을 순서대로 실행하세요. Colab CUDA 빌드 및 추론과 ROS–MuJoCo 물리 파지는 사용자가 실행합니다. score/width/depth 및 모델 방향을 보정하지 않습니다. grasp는 camera→object 좌표 변환만 수행합니다.


## 1. GPU와 CUDA 확인


In [ ]:
import sys, os, subprocess, pathlib
import torch
from torch.utils.cpp_extension import CUDA_HOME
print('Python:', sys.version)
print('PyTorch:', torch.__version__, 'torch CUDA:', torch.version.cuda, 'CUDA_HOME:', CUDA_HOME)
if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임을 선택한 뒤 다시 실행하세요.')
if CUDA_HOME is None:
    raise RuntimeError('CUDA compiler가 없는 런타임입니다.')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([str(pathlib.Path(CUDA_HOME)/'bin/nvcc'), '--version'], check=True)


## 2. 공식 baseline 확보 및 PointNet2 빌드
Colab의 기존 PyTorch를 사용합니다. 설치에 수분이 걸릴 수 있습니다. Torch/CUDA/compiler 조합의 호환성은 이 셀에서 확인해야 합니다.

변경 사항을 명시합니다: PointNet2의 deprecated C++ CUDA/type/pointer 호출만 호환 문법으로 수정합니다. KNN은 inference 경로에서 쓰지 않는 training-label 기능이지만 import되어, 이 노트북에서는 호출 시 오류를 내는 inference-only guard를 등록합니다. KNN 계산을 대체하거나 학습을 수행하지 않습니다. 네트워크/weights는 변경하지 않습니다. 실제 commit과 변경 파일을 출력/결과에 기록합니다. GraspNet API 패키지를 설치하지 않고 공식 pred_decode의 17열 출력을 그대로 export합니다.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ninja', 'scipy', 'matplotlib', 'gdown'], check=True)
REPO = pathlib.Path('/content/graspnet-baseline')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/graspnet/graspnet-baseline.git', str(REPO)], check=True)
BASELINE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
PATCHED_FILES = []
for path in sorted((REPO/'pointnet2/_ext_src').rglob('*')):
    if path.suffix not in ('.h', '.cpp', '.cu'):
        continue
    old = path.read_text()
    new = old.replace('.type().is_cuda()', '.is_cuda()').replace('.data<', '.data_ptr<')
    if new != old:
        path.write_text(new)
        PATCHED_FILES.append(str(path.relative_to(REPO)))
print('Baseline commit:', BASELINE_COMMIT, 'compatibility edits:', PATCHED_FILES)
os.environ['MAX_JOBS'] = '2'
major, minor = torch.cuda.get_device_capability(0)
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '--no-deps', str(REPO/'pointnet2')], check=True)
# Load the installed extension before the source tree can shadow its namespace.
import pointnet2._ext as pointnet_extension
import types
knn_guard = types.ModuleType('knn_modules')
def training_knn_unavailable(*args, **kwargs):
    raise RuntimeError('Training KNN was called: this notebook supports inference only.')
knn_guard.knn = training_knn_unavailable
sys.modules['knn_modules'] = knn_guard
for directory in ('models', 'utils', 'pointnet2'):
    sys.path.insert(0, str(REPO/directory))
from graspnet import GraspNet, pred_decode
print('PointNet2 extension and GraspNet imports OK')


## 3. 한 시점 입력 업로드
`data/pointclouds/cube_single_view/cube_single_view_input.zip`을 업로드합니다. 기본 카메라는 object frame [-0.50, 0, 0] m이며, 큐브 중심을 바라봅니다. 바닥면과 숨은 면은 제외합니다. 전체 표면 cloud와 다른 입력입니다.


In [ ]:
from google.colab import files
import io, zipfile, json, hashlib
import numpy as np
uploaded = files.upload()
archives = [value for name, value in uploaded.items() if name.endswith('.zip')]
if len(archives) != 1:
    raise ValueError('colab_input.zip 한 개를 업로드하세요.')
with zipfile.ZipFile(io.BytesIO(archives[0])) as archive:
    cloud_bytes = archive.read('points.npy')
    metadata = json.loads(archive.read('metadata.json'))
if hashlib.sha256(cloud_bytes).hexdigest() != metadata['npy_sha256']:
    raise ValueError('입력 파일과 metadata의 hash가 다릅니다.')
if metadata['units'] != 'meters' or metadata['source_frame'] != 'object':
    raise ValueError('meters/object frame 입력만 지원합니다.')
cloud = np.load(io.BytesIO(cloud_bytes), allow_pickle=False)
if cloud.shape != (metadata['num_points'], 3) or not np.isfinite(cloud).all():
    raise ValueError('유한한 N×3 좌표와 metadata가 필요합니다.')
if len(cloud) < 2048:
    raise ValueError('PointNet2용 입력 점이 너무 적습니다.')
print(metadata)
print('Bounds [m]:', cloud.min(axis=0), cloud.max(axis=0))

if metadata.get('view_policy') != 'one virtual viewpoint; visible faces only; support face excluded':
    raise ValueError(f"cube_single_view_input.zip을 선택하세요. 업로드한 view_policy={metadata.get('view_policy')!r}")
T_camera_object = np.asarray(metadata['T_camera_object'], dtype=float)
if T_camera_object.shape != (4,4) or not np.isfinite(T_camera_object).all():
    raise ValueError('Invalid camera transform')
if not np.allclose(T_camera_object[3], [0,0,0,1]) or not np.allclose(T_camera_object[:3,:3].T@T_camera_object[:3,:3], np.eye(3), atol=1e-6) or not np.isclose(np.linalg.det(T_camera_object[:3,:3]), 1):
    raise ValueError('Invalid rigid transform')
if not np.all(np.abs(cloud) <= .020001):
    raise ValueError('4 cm cube geometry invalid')
cloud_camera = (cloud@T_camera_object[:3,:3].T+T_camera_object[:3,3]).astype(np.float32)
print('Camera depth:', cloud_camera[:,2].min(), cloud_camera[:,2].max())


## 4. 공식 RealSense checkpoint 다운로드
공식 README의 Google Drive checkpoint-rs.tar를 사용합니다. 다운로드가 제한되면 [공식 링크](https://drive.google.com/file/d/1hd0G8LN6tRpi4742XOTEisbTXNZ-1jmk/view)에서 파일을 내려받아 이 셀의 업로드 창에서 선택하세요. `.tar`는 여기서 압축 해제하지 않고 torch checkpoint로 읽습니다.


In [ ]:
import gdown
CHECKPOINT = pathlib.Path('/content/checkpoint-rs.tar')
if not CHECKPOINT.exists():
    try:
        gdown.download(id='1hd0G8LN6tRpi4742XOTEisbTXNZ-1jmk', output=str(CHECKPOINT), quiet=False)
    except Exception as error:
        print('자동 다운로드 실패:', error)
if not CHECKPOINT.exists() or CHECKPOINT.stat().st_size < 1_000_000:
    print('공식 checkpoint-rs.tar를 직접 업로드하세요.')
    selected = files.upload()
    if 'checkpoint-rs.tar' not in selected:
        raise ValueError('checkpoint-rs.tar 파일이 필요합니다.')
    CHECKPOINT.write_bytes(selected['checkpoint-rs.tar'])
CHECKPOINT_SHA256 = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
print('Checkpoint bytes:', CHECKPOINT.stat().st_size, 'SHA256:', CHECKPOINT_SHA256)


## 5. 추론 및 후보 저장
모든 decoded 후보를 보존하고 JSON은 점수 내림차순으로 저장합니다. 후보가 없으면 명시적으로 실패합니다. 점수는 성공 확률로 해석하지 않습니다. 이 단계에서는 로봇·책상·물체 충돌 필터링과 NMS를 적용하지 않습니다.


In [ ]:
SEED = 42
NUM_INPUT_POINTS = 20000
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
rng = np.random.default_rng(SEED)
indices = rng.choice(len(cloud), NUM_INPUT_POINTS, replace=len(cloud) < NUM_INPUT_POINTS)
model_cloud = cloud_camera[indices].astype(np.float32)
# Network hyperparameters match the official pretrained baseline.
network = GraspNet(input_feature_dim=0, num_view=300, num_angle=12, num_depth=4,
                  cylinder_radius=0.05, hmin=-0.02, hmax_list=[0.01,0.02,0.03,0.04],
                  is_training=False).cuda().eval()
# weights_only=False is required for legacy official checkpoints. Use the official file only.
checkpoint = torch.load(str(CHECKPOINT), map_location='cpu', weights_only=False)
network.load_state_dict(checkpoint['model_state_dict'], strict=True)
with torch.inference_mode():
    prediction = network({'point_clouds': torch.from_numpy(model_cloud[None]).cuda()})
    candidates = pred_decode(prediction)[0].cpu().numpy()
if candidates.ndim != 2 or candidates.shape[1] != 17 or len(candidates) == 0:
    raise RuntimeError(f'NO_GRASP_CANDIDATE or unexpected shape: {candidates.shape}')
if not np.isfinite(candidates).all():
    raise RuntimeError('NONFINITE_GRASP_OUTPUT')
# Preserve original camera-frame decoded output, then perform only a rigid frame conversion.
candidates_camera_raw = candidates.copy()
T_object_camera = np.linalg.inv(T_camera_object)
candidates[:,4:13] = np.einsum('ij,njk->nik', T_object_camera[:3,:3], candidates[:,4:13].reshape(-1,3,3)).reshape(-1,9)
candidates[:,13:16] = candidates[:,13:16]@T_object_camera[:3,:3].T+T_object_camera[:3,3]
# Official layout: score,width,height,depth,R(row-major 9),translation(3),object_id.
rotations = candidates[:,4:13].reshape(-1,3,3)
if not np.allclose(rotations.transpose(0,2,1) @ rotations, np.eye(3), atol=1e-3) or not np.allclose(np.linalg.det(rotations), 1, atol=1e-3):
    raise RuntimeError('INVALID_ROTATION_MATRIX')
records = []
for index in np.argsort(-candidates[:,0], kind='stable'):
    row = candidates[index]
    records.append({'candidate_id': int(index), 'score': float(row[0]), 'width': float(row[1]),
                    'height': float(row[2]), 'depth': float(row[3]),
                    'rotation_matrix': row[4:13].reshape(3,3).tolist(),
                    'translation': row[13:16].tolist(), 'object_id': int(row[16])})
output = pathlib.Path('/content/graspnet_output')
output.mkdir(exist_ok=True)
np.save(output/'grasps_raw.npy', candidates, allow_pickle=False)
payload = {'schema_version': 1, 'object': metadata['object'], 'point_cloud_frame': 'object',
           'units': 'meters', 'rotation_convention': 'active; column vectors; R maps grasp to object',
           'approach_axis': '+X_grasp (rotation_matrix first column)',
           'approach_axis_source': 'baseline utils/loss_utils.py batch_viewpoint_params_to_matrix',
           'tcp_mapping': 'NOT_DEFINED', 'feasibility_checked': False,
           'filters_applied': [], 'input_metadata': metadata,
           'provenance': {'repository': 'https://github.com/graspnet/graspnet-baseline',
                          'commit': BASELINE_COMMIT, 'patched_files': PATCHED_FILES,
                          'knn': 'inference-only import guard; training call raises',
                          'checkpoint_sha256': CHECKPOINT_SHA256,
                          'checkpoint': 'official checkpoint-rs.tar',
                          'python': sys.version, 'torch': torch.__version__,
                          'cuda': torch.version.cuda, 'seed': SEED,
                          'num_input_points': NUM_INPUT_POINTS, 'inference_frame': 'virtual_camera',
                          'T_camera_object': T_camera_object.tolist(),
                          'view_policy': metadata['view_policy'],
                          'export_frame': 'object; inverse rigid transform only'},
           'grasps': records}
(output/'grasps.json').write_text(json.dumps(payload, indent=2, allow_nan=False)+'\n')
print('Candidates:', len(records))
print('Top candidate:', records[0])
print('These are unverified candidates, not an executable PiPER grasp.')

np.save(output/'grasps_camera_raw.npy', candidates_camera_raw, allow_pickle=False)
np.save(output/'network_input_camera.npy', model_cloud, allow_pickle=False)
# Geometric pre-grasp in object frame: same orientation, 20 cm opposite +X_grasp.
pre_records = []
for grasp in records:
    R = np.array(grasp['rotation_matrix']); p = np.array(grasp['translation'])
    T = np.eye(4); T[:3,:3] = R; T[:3,3] = p
    pre = T.copy(); pre[:3,3] -= .20*R[:,0]
    pre_records.append({**grasp, 'T_object_grasp': T.tolist(), 'T_object_pregrasp': pre.tolist()})
pre_payload = {**payload, 'grasps': pre_records, 'pregrasp_distance_m': .20,
               'pregrasp_frame': 'object', 'selected_candidate_id': None}
(output/'pregrasps_object.json').write_text(json.dumps(pre_payload, indent=2, allow_nan=False)+'\n')
print('Width <=70 mm:', sum(0 < g['width'] <= .07 for g in records), '/', len(records))
print('No score/width/depth/rotation adjustment; no top-down direction restriction.')


## 6. 입력 cloud와 상위 10개 후보 프레임 시각화
빨강=X(approach), 초록=Y, 파랑=Z. 표시된 프레임은 후보 grasp frame이며 PiPER TCP가 아닙니다. 단위는 m입니다.


In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(9,8))
axis = fig.add_subplot(111, projection='3d')
axis.scatter(*cloud[::10].T, s=1, c='gray', alpha=0.4)
for grasp in records[:10]:
    origin = np.asarray(grasp['translation'])
    rotation = np.asarray(grasp['rotation_matrix'])
    pre = origin-.20*rotation[:,0]
    axis.plot(*np.stack([pre,origin]).T, color='cyan', alpha=.3)
    for dimension, color in enumerate(('r','g','b')):
        vector = 0.03*rotation[:,dimension]
        axis.quiver(*origin, *vector, color=color)
pre_positions = candidates[:,13:16]-.20*candidates[:,4:13].reshape(-1,3,3)[:,:,0]
shown = np.concatenate([cloud, candidates[:,13:16], pre_positions])
low = shown.min(axis=0)
high = shown.max(axis=0)
center, extent = (low+high)/2, max(float((high-low).max()), 0.06)*0.7
axis.set_xlim(center[0]-extent,center[0]+extent)
axis.set_ylim(center[1]-extent,center[1]+extent)
axis.set_zlim(center[2]-extent,center[2]+extent)
axis.set_box_aspect((1,1,1))
axis.set_xlabel('X_object [m]'); axis.set_ylabel('Y_object [m]'); axis.set_zlabel('Z_object [m]')
fig.savefig(output/'candidate_frames.png', dpi=150)
plt.show()


## 7. 결과 다운로드
`cube_single_view_graspnet_output.zip`을 다운로드하세요. 후보와 기하학적 pre-grasp이며, TCP 변환/실행 가능 여부는 로컬 실행기가 검사합니다.


In [ ]:
result_zip = pathlib.Path('/content')/(metadata['object']+'_single_view_graspnet_output.zip')
with zipfile.ZipFile(result_zip,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ('grasps.json','grasps_raw.npy','candidate_frames.png','grasps_camera_raw.npy','network_input_camera.npy','pregrasps_object.json'):
        archive.write(output/name,arcname=name)
files.download(str(result_zip))
